In [ ]:
from tidalseis.load.network import read_network_traces, link_station_trace_paths, flatten_station_traces
from tidalseis.load.models import is_time_between
from network_catalog import AMERY_ICE_SHELF
from obspy.core import Stream, Trace, UTCDateTime  # type: ignore
from obspy import read
from obspy.signal.trigger import coincidence_trigger

from pathlib import Path
from datetime import datetime, timedelta
import numpy as np

import tidalseis._obspy_validation as vld

import matplotlib.pyplot as plt
import cmap
import matplotlib.colors as mcolor

# plt.switch_backend("qtagg")

DATE_FMT = ""
TEST_START = AMERY_ICE_SHELF["network_start"]
TEST_END = TEST_START + timedelta(days=4)
BASE = Path("D:/seismic_data/amery_ice_shelf/trace_data/")

In [ ]:
station_traces = read_network_traces(BASE)
station_traces_linked = link_station_trace_paths(station_traces, BASE)
all_traces = flatten_station_traces(station_traces_linked)
trace_dict: dict[str, Trace] = {}
super_stream = Stream()
print("Reading Traces...")
for i in all_traces:
    if not is_time_between(TEST_START, TEST_END, i.start):
        continue
    stream = vld.validate_stream(read(i.path))
    if stream.count() > 1:
        raise ValueError("Single trace streams for now...")
    trace = vld.validate_trace(stream.traces[0])
    trace.normalize()
    super_stream += trace
    if trace_dict.get(trace.id) is None:
        trace_dict[trace.id] = trace
    else:
        trace_dict[trace.id] += trace
    # print(
    #     f"{i.path.parent.name} --> {i.start} --> {trace.count()} samples"
    # )
print(f"Super stream contains {super_stream.count()} traces")

In [ ]:
print("Running coincidence trigger...")
triggers = coincidence_trigger(
    trigger_type="classicstalta",
    thr_on=6,
    thr_off=5,
    stream=super_stream.copy(),
    thr_coincidence_sum=2,
    lta=60,
    sta=2,
    details=True
)

In [ ]:
def trim_trace(trace: Trace, start: UTCDateTime, end: UTCDateTime) -> Trace:
    trimmed = trace.copy()

    stats = vld.validate_stats(trace.stats)
    rel_start = start - stats["starttime"]
    rel_end = end - stats["starttime"]
    print(start, stats["starttime"])
    start_idx = np.argmin(np.abs(trace.times()-rel_start))
    end_idx = np.argmin(np.abs(trace.times()- rel_end))
    trimmed.data = trace.data[start_idx:end_idx]

    return trimmed


In [ ]:
event_list: list[tuple[datetime, list[np.ndarray]]] = []
clr_map = cmap.Colormap("crameri:hawaii").to_mpl()
event = 0
for trig_dict in triggers:
    trig_dict = vld.validate_trigger(trig_dict)
    if trig_dict["duration"] > 120:
        continue
    print("")
    f, ax = plt.subplots()

    event_data: list[np.ndarray] = []
    norm = mcolor.Normalize(0, len(trig_dict["trace_ids"]))
    for n, id in enumerate(trig_dict["trace_ids"]):
        trace = trace_dict[id]
        trig_start = trig_dict["time"]
        trig_end = trig_dict["time"] + trig_dict["duration"]
        trig_trace = trace.slice(starttime=trig_start-1, endtime=trig_end)
        trig_trace.normalize()
        ax.plot(trig_trace.times(), trig_trace.data + n, color=clr_map(norm(n)))
        event_data.append(np.stack((trig_trace.times(),trig_trace.data), axis=-1))
    
    event_time: datetime = trig_dict["time"].datetime
    event_list.append((event_time, event_data))
    ax.set_title(f"Event: {event+1} @ {event_time.strftime("%B %d, %Y // %I:%M:%S%p")}")
    event+=1

# Add stations to metadata
print(len(event_list))
plt.show()